# Module 3 · Lab 1 — Build a Graph with LangGraph

LangGraph models an agent as a **graph**: a **State** flows through **Nodes** (Python functions) joined by **Edges** (which decide what runs next).

In this lab we build a graph from scratch with the **5 steps**, run it, then swap in a real LLM.

> First: run `uv sync` in this folder and select the **`3_langgraph/.venv`** kernel (top-right).

In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
import random

In [ ]:
# A couple of silly word lists — only for our first, LLM-free node
nouns = ["Cabbages", "Unicorns", "Toasters", "Penguins", "Bananas", "Zombies", "Rainbows", "Eels", "Pickles", "Muffins"]
adjectives = ["outrageous", "smelly", "pedantic", "existential", "moody", "sparkly", "untrustworthy", "sarcastic", "squishy", "haunted"]

In [ ]:
# Load OPENAI_API_KEY (and friends) from your .env file
load_dotenv(override=True)

### A word about `Annotated` and reducers

When a node returns a new state, how should LangGraph **combine** it with the old one? You tell it, per field, with a **reducer** — declared using `Annotated`.

LangGraph ships one for message lists: **`add_messages`**, which appends new messages onto the list.

### Step 1 — define the State

Our graph carries one thing: a list of messages.

In [ ]:
class State(BaseModel):
    messages: Annotated[list, add_messages]

### Step 2 — start the Graph Builder

Pass the State **class** (not an instance).

In [ ]:
graph_builder = StateGraph(State)

### Step 3 — create a node

A node is just a Python function: it takes the old state and returns a new one. To prove a node doesn't *need* an LLM, this first one returns a random silly sentence.

In [ ]:
def our_first_node(old_state: State) -> State:
    reply = f"{random.choice(nouns)} are {random.choice(adjectives)}"
    messages = [{"role": "assistant", "content": reply}]
    return State(messages=messages)

graph_builder.add_node("first_node", our_first_node)

### Step 4 — create edges

`START` and `END` are built-in markers.

In [ ]:
graph_builder.add_edge(START, "first_node")
graph_builder.add_edge("first_node", END)

### Step 5 — compile the graph

Now it's runnable — and LangGraph can draw it for us.

In [ ]:
graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### Showtime — run it

We wrap `graph.invoke(state)` in a Gradio chat. It only returns silly sentences — that's the point: a node is a plain function.

In [ ]:
def chat(user_input: str, history):
    state = State(messages=[{"role": "user", "content": user_input}])
    result = graph.invoke(state)
    return result["messages"][-1].content

gr.ChatInterface(chat, type="messages").launch()

### Now with a real LLM

Same 5 steps — but the node calls an LLM. `ChatOpenAI` is LangChain's wrapper. We do all 5 in a few cells.

In [ ]:
# Steps 1 + 2
class State(BaseModel):
    messages: Annotated[list, add_messages]

graph_builder = StateGraph(State)

In [ ]:
# Step 3 — a node that calls the LLM
llm = ChatOpenAI(model="gpt-5.4-mini")

def chatbot_node(old_state: State) -> State:
    response = llm.invoke(old_state.messages)
    return State(messages=[response])

graph_builder.add_node("chatbot", chatbot_node)

In [ ]:
# Steps 4 + 5 — edges, compile, draw
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

graph = graph_builder.compile()
display(Image(graph.get_graph().draw_mermaid_png()))

### Talk to it

Now it's a real chatbot. But try: tell it your name, then ask what your name is — it won't remember. **Every `invoke` is a fresh run; there's no memory yet.** We fix that in Lab 2.

In [ ]:
def chat(user_input: str, history):
    state = State(messages=[{"role": "user", "content": user_input}])
    result = graph.invoke(state)
    return result["messages"][-1].content

gr.ChatInterface(chat, type="messages").launch()